<div align="center">

<img src="https://raw.githubusercontent.com/varaslaw/ultimate-aisingers/main/src/ultimate_rvc/web/assets/aisingers-hero.png" width="720" alt="AISingers"/>

# AISingers Studio для Kaggle
### AI-каверы и озвучка · русская версия

[Наш Telegram-канал](https://t.me/aisingers) · [Создать голосового бота](https://t.me/AIsingers_bot) · [GitHub](https://github.com/varaslaw/ultimate-aisingers)

</div>


## Перед запуском

В настройках Kaggle включите **Accelerator: GPU** и **Internet: On**. Затем последовательно выполните все ячейки. Проект устанавливается в отдельное окружение Python 3.12, поэтому встроенный Gradio Kaggle не конфликтует с AISingers.


In [ ]:
# Шаг 1 · Подготовка и загрузка проекта
import os
import shutil
from pathlib import Path
from IPython.display import clear_output

PROJECT_DIR = Path("/kaggle/working/AISingers").resolve()
EXPECTED_ROOT = Path("/kaggle/working").resolve()
repo_branch = "main"
fresh_clone = True

if fresh_clone and PROJECT_DIR.exists():
    if PROJECT_DIR.parent != EXPECTED_ROOT or PROJECT_DIR.name != "AISingers":
        raise RuntimeError(f"Небезопасный путь проекта: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    !git clone --depth 1 --branch {repo_branch} https://github.com/varaslaw/ultimate-aisingers {PROJECT_DIR}

%cd /kaggle/working/AISingers
clear_output()
print("✓ AISingers загружен из ветки", repo_branch)


## Шаг 2 · Совместимое окружение

Эта ячейка не меняет системный Python Kaggle. Она создаёт изолированное окружение AISingers и фиксирует совместимую версию Gradio. Первый запуск может занять несколько минут.


In [ ]:
# Шаг 2 · Установка зависимостей
!curl -LsSf https://astral.sh/uv/0.9.11/install.sh | sh
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ["URVC_CONSOLE_LOG_LEVEL"] = "WARNING"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

!uv python install 3.12
!uv sync --python 3.12 --extra cuda --prerelease if-necessary-or-explicit
!uv pip install --python .venv/bin/python "gradio==5.49.1" "matplotlib-inline==0.1.7"
!uv run python ./src/ultimate_rvc/core/main.py
clear_output()
print("✓ Зависимости установлены в изолированное окружение")


In [ ]:
# Шаг 3 · Проверка окружения
!uv run python -c "import gradio, torch; print('Gradio:', gradio.__version__); print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.is_available()); print('Видеокарта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найдена')"


## Шаг 4 · Запуск интерфейса

По умолчанию используется Cloudflare-туннель — он обычно стабильнее в Kaggle. Если хотите стандартную ссылку `gradio.live`, замените значение `publish_method` на `gradio`. Не останавливайте ячейку, пока пользуетесь студией.


In [ ]:
# Шаг 4 · Публикация AISingers
import re
import subprocess
import time
from urllib.request import urlretrieve

publish_method = "cloudflared"  # варианты: cloudflared, gradio
port = 7860
run_path = "./src/ultimate_rvc/web/main.py"

if publish_method == "gradio":
    !uv run python {run_path} --share --listen-port {port}
elif publish_method == "cloudflared":
    cloudflared = Path("/kaggle/working/cloudflared")
    if not cloudflared.exists():
        urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            cloudflared,
        )
        cloudflared.chmod(0o755)

    server_log = Path("/kaggle/working/aisingers-server.log")
    tunnel_log = Path("/kaggle/working/aisingers-tunnel.log")
    with server_log.open("w", encoding="utf-8") as log:
        server = subprocess.Popen(
            ["uv", "run", "python", run_path, "--listen", "--listen-port", str(port)],
            stdout=log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR,
        )
    time.sleep(8)
    with tunnel_log.open("w", encoding="utf-8") as log:
        tunnel = subprocess.Popen(
            [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=log, stderr=subprocess.STDOUT,
        )

    public_url = None
    for _ in range(30):
        time.sleep(1)
        text = tunnel_log.read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if match:
            public_url = match.group(0)
            break

    if not public_url:
        print(tunnel_log.read_text(encoding="utf-8", errors="ignore")[-3000:])
        raise RuntimeError("Не удалось получить Cloudflare-ссылку. Проверьте Internet: On.")
    print("\n✓ AISingers запущен:", public_url)
else:
    raise ValueError("publish_method должен быть cloudflared или gradio")


---

### Сообщество AISingers

- [Telegram-канал AISingers](https://t.me/aisingers) — новости, обновления и материалы.
- [@AIsingers_bot](https://t.me/AIsingers_bot) — создание голосовых ботов в Telegram.
